In [1]:
import pandas as pd
from pathlib import Path

# Los CSV originales de Olist están junto a esta nota.
opciones = [
    Path.cwd() / "datos_olist_reales",
    Path.cwd() / "estudio_pandas" / "examen_olist" / "datos_olist_reales",
    Path.cwd().parent / "datos_olist_reales",
]
carpeta = next((p for p in opciones if (p / "olist_orders_dataset.csv").is_file()), None)
if carpeta is None:
    raise FileNotFoundError(
        "No encuentro datos_olist_reales. Abre esta nota desde examen_olist "
        "o ajusta la variable carpeta a su ubicación."
    )

orders = pd.read_csv(carpeta / "olist_orders_dataset.csv")
customers = pd.read_csv(carpeta / "olist_customers_dataset.csv")
order_items = pd.read_csv(carpeta / "olist_order_items_dataset.csv")
products = pd.read_csv(carpeta / "olist_products_dataset.csv")
sellers = pd.read_csv(carpeta / "olist_sellers_dataset.csv")
order_payments = pd.read_csv(carpeta / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(carpeta / "olist_order_reviews_dataset.csv")
translation = pd.read_csv(carpeta / "product_category_name_translation.csv")

# Convertimos solo las fechas necesarias en estos ejercicios.
# Distinguimos los nulos originales de los valores invalidados por la conversión.
for columna in [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    original = orders[columna]
    convertido = pd.to_datetime(
        original, format="%Y-%m-%d %H:%M:%S", errors="coerce"
    )
    nuevos_nat = (original.notna() & convertido.isna()).sum()
    print(columna, "nulos originales:", original.isna().sum(),
          "nuevos NaT:", nuevos_nat)
    orders[columna] = convertido

for nombre in ["orders", "customers", "order_items", "products", "sellers",
               "order_payments", "order_reviews", "translation"]:
    tabla = globals()[nombre]
    print(f"{nombre}: {len(tabla):,} filas × {len(tabla.columns)} columnas")

    

order_purchase_timestamp nulos originales: 0 nuevos NaT: 0
order_delivered_customer_date nulos originales: 2965 nuevos NaT: 0
order_estimated_delivery_date nulos originales: 0 nuevos NaT: 0
orders: 99,441 filas × 8 columnas
customers: 99,441 filas × 5 columnas
order_items: 112,650 filas × 7 columnas
products: 32,951 filas × 9 columnas
sellers: 3,095 filas × 4 columnas
order_payments: 103,886 filas × 5 columnas
order_reviews: 99,224 filas × 7 columnas
translation: 71 filas × 2 columnas


## Notas Clase 
- 16:12hrs

El EDA es clave para el anális laboral, se suelen hacer en Notebooks

### Para el examen final, se nos da una base de datos, limpiarla, acomodarla, analaizarla etc.
- Es clave hacer una observación de como funciona (en términos comprensibles)
`histograma`, `mapas de calor`, `gráficos de barra` ,`gráficos de dispersión`, etc.



### Temas pendientes de Pandas del parcial pasado
- `Tablas pivóte`
- `funciones lambda`

### Sintáxis de una tabla pivóte

df.pivot_table(
    values=
    index=
    columns=
    agg_func =
)

# Funciones lambda



In [2]:
# Forma normal
def cuadrados(x):
    return x ** 2

#Con lambda
cuadrados_1 = lambda x: x ** 2

#Ejemplo funcion 

def clasificar(x):
    if x > 100:
        return "caro"
    else:
        return "barato"

#df("es_mueble") = df["product_category_name"].apply(
#clasificar
#)


# Aqui hare los problemas de Pivot tables y lambda



In [3]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [4]:
# 1. 
order_items.info()



r1 =order_items.pivot_table(
    values = "price",
    index= "product_id",
    aggfunc= "mean"

)

display(r1)

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


,price
product_id,
00066f42aeeb9f3007548bb9d3f33c38,101.65
00088930e925c41fd95ebfe695fd2655,129.90
0009406fd7479715e4bef61dd91f2462,229.00
000b8f95fcb9e0096488278317764d19,58.90
000d9be29b5207b54e86aa1b1ac54872,199.00
...,...
fff6177642830a9a94a0f2cba5e476d1,114.99
fff81cc3158d2725c0655ab9ba0f712c,90.00
fff9553ac224cec9d15d49f5a263411f,32.00


In [5]:
#2 

r2 = order_items.pivot_table(
    values= "price",
    index= "product_id",
    aggfunc= "sum"
)

x = r1 != r2
y = x["price"].sum()
print(y)

display(r2)

14834


,price
product_id,
00066f42aeeb9f3007548bb9d3f33c38,101.65
00088930e925c41fd95ebfe695fd2655,129.90
0009406fd7479715e4bef61dd91f2462,229.00
000b8f95fcb9e0096488278317764d19,117.80
000d9be29b5207b54e86aa1b1ac54872,199.00
...,...
fff6177642830a9a94a0f2cba5e476d1,229.98
fff81cc3158d2725c0655ab9ba0f712c,90.00
fff9553ac224cec9d15d49f5a263411f,32.00


In [6]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


# A partir de aquí hare la tarea empezando desde el Nivel 2 












 

In [7]:
#6 Construye una pivot table donde las filas representen la categoría del producto y las columnas representen el estado del pedido. Muestra el precio total.
order_items.info()
items_con_productos = pd.merge(order_items, products, on='product_id', how='left')
display(items_con_productos)
items_con_productos_y_ordenes = pd.merge(items_con_productos, orders, on='order_id', how='left')

r6 = pd.pivot_table(
    data= items_con_productos_y_ordenes,
    index='product_category_name',
    columns= 'order_status',
    values= 'price',
    aggfunc= 'sum'
)

display(r6)

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,utilidades_domesticas,43.0,1002.0,3.0,10150.0,89.0,15.0,40.0
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,informatica_acessorios,31.0,232.0,1.0,8950.0,45.0,26.0,38.0
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,esporte_lazer,43.0,869.0,1.0,967.0,21.0,24.0,19.0
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,informatica_acessorios,56.0,1306.0,1.0,100.0,20.0,20.0,20.0


order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
product_category_name,,,,,,,
agro_industria_e_comercio,NaN,NaN,70566.10,19.50,NaN,1944.87,NaN
alimentos,NaN,149.90,28731.15,NaN,221.60,290.76,NaN
alimentos_bebidas,NaN,60.00,14942.88,62.80,8.00,105.80,NaN
artes,NaN,35.00,23657.30,NaN,224.90,285.44,NaN
artes_e_artesanato,NaN,NaN,1814.01,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
sinalizacao_e_seguranca,NaN,NaN,21315.05,NaN,139.90,54.28,NaN
tablets_impressao_imagem,NaN,NaN,7528.41,NaN,NaN,NaN,NaN
telefonia,NaN,1324.89,309860.23,4654.13,4475.14,3353.14,NaN


In [8]:
#7 Repite el ejercicio anterior utilizando el promedio del precio.

r7= pd.pivot_table(
    data = items_con_productos_y_ordenes,
    index = 'product_category_name',
    columns= 'order_status',
    values= 'price',
    aggfunc= 'mean'
)

display(r7)

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
product_category_name,,,,,,,
agro_industria_e_comercio,NaN,NaN,342.553883,19.500000,NaN,388.974000,NaN
alimentos,NaN,149.900000,57.577455,NaN,44.320000,58.152000,NaN
alimentos_bebidas,NaN,30.000000,55.549740,15.700000,8.000000,52.900000,NaN
artes,NaN,35.000000,120.087817,NaN,32.128571,71.360000,NaN
artes_e_artesanato,NaN,NaN,75.583750,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
sinalizacao_e_seguranca,NaN,NaN,108.198223,NaN,139.900000,54.280000,NaN
tablets_impressao_imagem,NaN,NaN,90.703735,NaN,NaN,NaN,NaN
telefonia,NaN,73.605000,69.945876,221.625238,248.618889,57.812759,NaN


In [9]:
#8 Construye una tabla que permita comparar el freight_value promedio entre categorías y estados del pedido.

r8 = pd.pivot_table(
    data= items_con_productos_y_ordenes,
    index= 'product_category_name',
    columns= 'order_status',
    values= 'freight_value',
    aggfunc= 'mean'

)

display(r8)

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
product_category_name,,,,,,,
agro_industria_e_comercio,NaN,NaN,27.365049,8.720000,NaN,39.536000,NaN
alimentos,NaN,17.760000,14.155371,NaN,15.614000,22.334000,NaN
alimentos_bebidas,NaN,9.685000,16.337881,11.850000,14.520000,15.905000,NaN
artes,NaN,16.600000,19.432640,NaN,19.108571,16.645000,NaN
artes_e_artesanato,NaN,NaN,15.422083,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
sinalizacao_e_seguranca,NaN,NaN,32.880914,NaN,18.550000,11.730000,NaN
tablets_impressao_imagem,NaN,NaN,14.773494,NaN,NaN,NaN,NaN
telefonia,NaN,14.022778,15.652910,12.056190,21.676667,16.855690,NaN


In [10]:
#9 Utiliza fill_value=0 y compara el resultado con la tabla original. ¿Qué combinaciones no tenían observaciones?

pivot_fill_0 = pd.pivot_table(
    data= items_con_productos_y_ordenes,
    columns= 'order_status',
    values='freight_value',
    aggfunc= 'mean',
    fill_value=0
)

combinaciones_vacias = r8.isna()

print("True = no hubo ventas en ese estado para esa categoría")
display(combinaciones_vacias)


True = no hubo ventas en ese estado para esa categoría


order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
product_category_name,,,,,,,
agro_industria_e_comercio,True,True,False,False,True,False,True
alimentos,True,False,False,True,False,False,True
alimentos_bebidas,True,False,False,False,False,False,True
artes,True,False,False,True,False,False,True
artes_e_artesanato,True,True,False,True,True,True,True
...,...,...,...,...,...,...,...
sinalizacao_e_seguranca,True,True,False,True,False,False,True
tablets_impressao_imagem,True,True,False,True,True,True,True
telefonia,True,False,False,False,False,False,True


In [11]:
#10 Construye una pivot table utilizando más de una variable en index. Interpreta qué representa cada nivel del índice.
items_pivot = items_con_productos_y_ordenes.copy()
items_pivot['product_category_name'] = items_pivot['product_category_name'].fillna('sin_categoria')

r10 = pd.pivot_table(
    data=items_pivot,
    index=['order_status', 'product_category_name'],
    values='price',
    aggfunc='sum'
)

display(r10)


price
order_status product_category_name          
approved     informatica_acessorios    39.70
             utilidades_domesticas    169.90
canceled     alimentos                149.90
             alimentos_bebidas         60.00
             artes                     35.00
...                                      ...
unavailable  automotivo                35.00
             bebes                    158.99
             cool_stuff                64.90
             eletronicos             1199.00
             perfumaria               549.80

[279 rows x 1 columns]

# Nivel 3

In [12]:
#11 Crea una columna llamada total que sea la suma de price y freight_value.

base_para_lambda = pd.merge(items_con_productos, orders, on='order_id',how='left')

base_para_lambda['total'] = base_para_lambda.apply(lambda row: row['price'] + row['freight_value'], axis=1)
display(base_para_lambda[['price', 'freight_value', 'total']].head())

,price,freight_value,total
0,58.90,13.29,72.19
1,239.90,19.93,259.83
2,199.00,17.87,216.87
3,12.99,12.79,25.78
4,199.90,18.14,218.04


In [13]:
#12 Crea una columna que clasifique los productos como "barato" o "caro" según un umbral que tú determines. Justifica el umbral.

umbral_precio = base_para_lambda['price'].median()

base_para_lambda['clasificacion_precio'] = base_para_lambda['price'].apply(
    lambda precio: 'caro' if precio > umbral_precio else 'barato'
)

display(base_para_lambda[['price','clasificacion_precio']])

#El umbral fue tomar la mediana del precio, porque nos da un valor aproximado del centro de la distribucion de precios

,price,clasificacion_precio
0,58.90,barato
1,239.90,caro
2,199.00,caro
3,12.99,barato
4,199.90,caro
...,...,...
112645,299.99,caro
112646,350.00,caro
112647,99.90,caro
112648,55.99,barato


In [14]:
#13 Crea una columna que indique si el costo de envió representa mas del 20% del precio del producto.

base_para_lambda["envio_mas_del_20_p"] = base_para_lambda.apply(
    lambda row : True if row["freight_value"] > (0.20 * row["price"]) else False,
    axis = 1
)

display(base_para_lambda[["price", "freight_value", "envio_mas_del_20_p"]])

,price,freight_value,envio_mas_del_20_p
0,58.90,13.29,True
1,239.90,19.93,False
2,199.00,17.87,False
3,12.99,12.79,True
4,199.90,18.14,False
...,...,...,...
112645,299.99,43.41,False
112646,350.00,36.53,False
112647,99.90,16.95,False
112648,55.99,8.72,False


In [15]:
#14 Crear una nueva columna que convierta el nombre de la categoria a mayúsculas

items_con_productos["categoria_MAYUSCULAS"] = items_con_productos["product_category_name"].apply(
    lambda cat: str(cat).upper() if pd.notnull(cat) else cat
)

display(items_con_productos[["product_category_name", "categoria_MAYUSCULAS"]])

,product_category_name,categoria_MAYUSCULAS
0,cool_stuff,COOL_STUFF
1,pet_shop,PET_SHOP
2,moveis_decoracao,MOVEIS_DECORACAO
3,perfumaria,PERFUMARIA
4,ferramentas_jardim,FERRAMENTAS_JARDIM
...,...,...
112645,utilidades_domesticas,UTILIDADES_DOMESTICAS
112646,informatica_acessorios,INFORMATICA_ACESSORIOS
112647,esporte_lazer,ESPORTE_LAZER
112648,informatica_acessorios,INFORMATICA_ACESSORIOS


In [16]:
#15 Crea una columna que indique si el nombre de la categoria contiene una palabra determinada.

palabra_clave= "cama"

items_con_productos["contiene_palabra"] = items_con_productos["product_category_name"].apply(
    lambda cat : palabra_clave in str(cat).lower() if pd.notnull(cat) else False
)

display(items_con_productos[["product_category_name", "contiene_palabra"]])

,product_category_name,contiene_palabra
0,cool_stuff,False
1,pet_shop,False
2,moveis_decoracao,False
3,perfumaria,False
4,ferramentas_jardim,False
...,...,...
112645,utilidades_domesticas,False
112646,informatica_acessorios,False
112647,esporte_lazer,False
112648,informatica_acessorios,False


# Nivel 4

In [17]:
#16 Crea una varaible que clasifique los pedidos según el costo total y después construye una pivot table con el costo total promedio por categoria y segmento

datos_totales = pd.merge(items_con_productos, orders, on="order_id", how="left")

#Usare el percentil 75 para los más caros
datos_totales["costo_total"] = datos_totales["price"] + datos_totales["freight_value"]
umbral_alto = datos_totales["costo_total"].quantile(0.75)

datos_totales["segmento_costo"] = datos_totales["costo_total"].apply(
    lambda costo: "Alto valor" if costo >= umbral_alto else "Valor regular"
)

r16 = pd.pivot_table(
    data= datos_totales,
    index="product_category_name",
    columns="segmento_costo",
    values= "costo_total",
    aggfunc= "mean"
)

display(r16)

segmento_costo,Alto valor,Valor regular
product_category_name,,
agro_industria_e_comercio,594.816992,58.557079
alimentos,189.170256,62.180042
alimentos_bebidas,261.000476,55.278054
artes,805.414545,97.920455
artes_e_artesanato,310.520000,71.050000
...,...,...
sinalizacao_e_seguranca,328.438519,70.906000
tablets_impressao_imagem,423.133333,93.565125
telefonia,338.414056,47.964726


In [18]:
#17 Calcula el porcentaje que representa el envió respecto al precio y crea una clasificación: "envio bajo" y "envio alto". Después commpara ambas categorias mediante un pivot table

#Usare 15% como umbral de envio alto

datos_totales["clasificacion_envio"] = datos_totales.apply(
    lambda row:"envio alto" if row["price"] > 0 and (row["freight_value"] / row["price"]) > 0.15 else "envio bajo",
    axis=1

)


datos_totales.columns
datos_totales.info()
r17 = pd.pivot_table(
    data= datos_totales,
    index = "product_category_name",
    columns="clasificacion_envio",
    values="order_id",
    aggfunc="count"

)

display(r17)



<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 27 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  str           
 1   order_item_id                  112650 non-null  int64         
 2   product_id                     112650 non-null  str           
 3   seller_id                      112650 non-null  str           
 4   shipping_limit_date            112650 non-null  str           
 5   price                          112650 non-null  float64       
 6   freight_value                  112650 non-null  float64       
 7   product_category_name          111047 non-null  str           
 8   product_name_lenght            111047 non-null  float64       
 9   product_description_lenght     111047 non-null  float64       
 10  product_photos_qty             111047 non-null  float64       
 11  product_wei

clasificacion_envio,envio alto,envio bajo
product_category_name,,
agro_industria_e_comercio,86.0,126.0
alimentos,419.0,91.0
alimentos_bebidas,255.0,23.0
artes,140.0,69.0
artes_e_artesanato,17.0,7.0
...,...,...
sinalizacao_e_seguranca,148.0,51.0
tablets_impressao_imagem,58.0,25.0
telefonia,3870.0,675.0


In [19]:
#18 Crea una variable que indique si el pedido tiene un precio superior al precio promedio de todos los productos. Después analiza esa variable mediante una pivot table

precio_promedio_global = datos_totales["price"].mean()

datos_totales["superior_al_promedio"] = datos_totales["price"].apply(
    lambda p: "Mayor al Promedio" if p > precio_promedio_global else "Menor o igual"
)

r18 = pd.pivot_table(
    data=datos_totales,
    index="order_status",
    columns="superior_al_promedio",
    values="price",
    aggfunc=["count", "sum"]
)

display(r18)

count                             sum  \
superior_al_promedio Mayor al Promedio Menor o igual Mayor al Promedio   
order_status                                                             
approved                             1             2            169.90   
canceled                           200           342          76932.10   
delivered                        31243         78954        8667427.55   
invoiced                           152           207          50267.11   
processing                         165           192          50090.66   
shipped                            364           821         103458.40   
unavailable                          3             4           1748.80   

                                    
superior_al_promedio Menor o igual  
order_status                        
approved                     39.70  
canceled                  18303.17  
delivered               4554070.56  
invoiced                  11259.26  
processing                10348.56  
shipped                   47269.04  
unavailable                 258.89

In [20]:
#19 Crea una pivot table que muestre, para cada categoria, el total y el promedio de una variable numérica que consideres relevante

#Variable: freight_value

r19 = pd.pivot_table(
    data= datos_totales,
    index = "product_category_name",
    values= "freight_value",
    aggfunc=["sum", "mean"]
)

display(r19)

,sum,mean
,freight_value,freight_value
product_category_name,,
agro_industria_e_comercio,5843.60,27.564151
alimentos,7271.03,14.256922
alimentos_bebidas,4507.99,16.215791
artes,4045.17,19.354880
artes_e_artesanato,370.13,15.422083
...,...,...
sinalizacao_e_seguranca,6507.82,32.702613
tablets_impressao_imagem,1226.20,14.773494


# Nivel 5

In [21]:
#20 pregunta, En qué categorías el flete pesa más respecto al precio, y cambia eso si el producto es barato o caro?

datos_ejercicio_20 = datos_totales.copy()
datos_ejercicio_20['product_category_name'] = datos_ejercicio_20['product_category_name'].fillna('sin_categoria')

#porcentaje que representa el flete sobre el precio

datos_ejercicio_20['porcentaje_flete'] = datos_ejercicio_20.apply(
    lambda fila: fila['freight_value'] / fila['price'] * 100, axis=1
)


precio_promedio = datos_ejercicio_20['price'].mean()
datos_ejercicio_20['tipo_producto'] = datos_ejercicio_20['price'].apply(
    lambda precio: 'Caro' if precio > precio_promedio else 'Barato'
)

#solo datos con al menos 500 items para que sea representativo

items_por_categoria = datos_ejercicio_20['product_category_name'].value_counts()
categorias_con_volumen = items_por_categoria[items_por_categoria >= 500].index
mascara_categorias = datos_ejercicio_20['product_category_name'].isin(categorias_con_volumen)
datos_ejercicio_20 = datos_ejercicio_20[mascara_categorias]

tabla_flete_por_categoria = pd.pivot_table(
    data=datos_ejercicio_20,
    index='product_category_name',
    columns='tipo_producto',
    values='porcentaje_flete',
    aggfunc='mean'
)

display(tabla_flete_por_categoria)



tipo_producto,Barato,Caro
product_category_name,,
alimentos,36.793097,13.794254
automotivo,44.288697,12.657116
bebes,35.678270,15.280308
beleza_saude,37.812998,10.372815
brinquedos,32.060239,12.199240
cama_mesa_banho,32.660169,13.553029
casa_construcao,32.517882,15.337422
consoles_games,49.228107,7.420075
construcao_ferramentas_construcao,49.169339,13.140344


In [22]:
#21 relacion entre precio, costo de envio y categoría

datos_ejercicio_21 = datos_totales.copy()
datos_ejercicio_21['product_category_name'] = datos_ejercicio_21['product_category_name'].fillna('sin_categoria')

#flete mas del 30% del precio

datos_ejercicio_21['flete_es_alto'] = datos_ejercicio_21.apply(
    lambda fila: 'Si' if fila['freight_value'] > fila['price'] * 0.3 else 'No', axis=1
)
#categorias con al menos 500 items(misma logica que el 20)

items_por_categoria = datos_ejercicio_21['product_category_name'].value_counts()
categorias_con_volumen = items_por_categoria[items_por_categoria >= 500].index
datos_ejercicio_21 = datos_ejercicio_21[datos_ejercicio_21['product_category_name'].isin(categorias_con_volumen)]

#precio promedio y flete promedio por categoria 
tabla_precio_y_flete = pd.pivot_table(
    data=datos_ejercicio_21,
    index='product_category_name',
    values=['price', 'freight_value'],
    aggfunc='mean'
)
display(tabla_precio_y_flete.round(2))

#cantidad de items con flete alto o no, por categoría
tabla_flete_alto = pd.pivot_table(
    data=datos_ejercicio_21,
    index='product_category_name',
    columns='flete_es_alto',
    values='price',
    aggfunc='count'
)
tabla_flete_alto['porcentaje_flete_alto'] = tabla_flete_alto['Si'] / (tabla_flete_alto['Si'] + tabla_flete_alto['No']) * 100
display(tabla_flete_alto.sort_values('porcentaje_flete_alto', ascending=False).round(1))

# Correlación entre precio y flete
print(datos_ejercicio_21[['price', 'freight_value']].corr())

,freight_value,price
product_category_name,,
alimentos,14.26,57.63
automotivo,21.88,139.96
bebes,22.30,134.34
beleza_saude,18.88,130.16
brinquedos,18.81,117.55
cama_mesa_banho,18.42,93.30
casa_construcao,22.90,137.56
consoles_games,17.44,138.49
construcao_ferramentas_construcao,22.23,155.73


flete_es_alto,No,Si,porcentaje_flete_alto
product_category_name,,,
eletronicos,728,2039,73.7
telefonia,1588,2957,65.1
eletrodomesticos,375,396,51.4
utilidades_domesticas,3652,3312,47.6
consoles_games,603,534,47.0
fashion_bolsas_e_acessorios,1117,914,45.0
moveis_sala,278,225,44.7
livros_interesse_geral,311,242,43.8
moveis_decoracao,4752,3582,43.0


                  price  freight_value
price          1.000000       0.407757
freight_value  0.407757       1.000000


In [23]:
#22 Reporte: categorías según el estado del pedido 
datos_ejercicio_22 = datos_totales.copy()
datos_ejercicio_22['product_category_name'] = datos_ejercicio_22['product_category_name'].fillna('sin_categoria')


datos_ejercicio_22['grupo_estado'] = datos_ejercicio_22['order_status'].apply(
    lambda estado: 'Problematico' if estado in ['canceled', 'unavailable'] else 'Normal'
)


items_por_categoria = datos_ejercicio_22['product_category_name'].value_counts()
categorias_con_volumen = items_por_categoria[items_por_categoria >= 500].index
datos_ejercicio_22 = datos_ejercicio_22[datos_ejercicio_22['product_category_name'].isin(categorias_con_volumen)]

# cantidad de items por categoría y grupo de estado
tabla_cantidad_items = pd.pivot_table(
    data=datos_ejercicio_22,
    index='product_category_name',
    columns='grupo_estado',
    values='price',
    aggfunc='count'
)
tabla_cantidad_items['porcentaje_problematico'] = (
    tabla_cantidad_items['Problematico'] / (tabla_cantidad_items['Problematico'] + tabla_cantidad_items['Normal']) * 100
)
display(tabla_cantidad_items.sort_values('porcentaje_problematico', ascending=False).round(2))

# precio promedio por categoría y grupo de estado
tabla_precio_promedio = pd.pivot_table(
    data=datos_ejercicio_22,
    index='product_category_name',
    columns='grupo_estado',
    values='price',
    aggfunc='mean'
)
display(tabla_precio_promedio)

grupo_estado,Normal,Problematico,porcentaje_problematico
product_category_name,,,
instrumentos_musicais,669,11,1.62
livros_interesse_geral,546,7,1.27
eletroportateis,671,8,1.18
consoles_games,1127,10,0.88
sin_categoria,1589,14,0.87
brinquedos,4083,34,0.83
automotivo,4204,31,0.73
bebes,3043,22,0.72
utilidades_domesticas,6915,49,0.70


grupo_estado,Normal,Problematico
product_category_name,,
alimentos,57.452868,149.900000
automotivo,139.530383,197.883226
bebes,134.838055,66.031364
beleza_saude,130.339955,82.950278
brinquedos,117.513725,121.707647
cama_mesa_banho,93.355327,56.923333
casa_construcao,137.661891,78.000000
consoles_games,137.468740,253.795000
construcao_ferramentas_construcao,155.933323,63.700000
